In [239]:
import pandas as pd
import numpy as np

In [240]:
df=pd.read_csv('brca_cancer (1).csv')

In [241]:
df.head()

,Unnamed: 0,gene_id,gene_name,gene_type,unstranded,stranded_first,stranded_second,tpm_unstranded,fpkm_unstranded,fpkm_uq_unstranded,class
0,4,ENSG00000000003.15,TSPAN6,protein_coding,5175,2582,2594,58.4397,16.4625,16.0650,0
1,5,ENSG00000000005.6,TNMD,protein_coding,85,52,33,2.9499,0.8310,0.8109,0
2,6,ENSG00000000419.13,DPM1,protein_coding,2406,1194,1212,102.1078,28.7639,28.0692,0
3,7,ENSG00000000457.14,SCYL3,protein_coding,2222,1629,1631,16.5362,4.6583,4.5458,0
4,8,ENSG00000000460.17,C1orf112,protein_coding,476,808,782,4.0842,1.1505,1.1227,0


In [242]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 121320 entries, 0 to 121319
Data columns (total 11 columns):
 #   Column              Non-Null Count   Dtype  
---  ------              --------------   -----  
 0   Unnamed: 0          121320 non-null  int64  
 1   gene_id             121320 non-null  object 
 2   gene_name           121320 non-null  object 
 3   gene_type           121320 non-null  object 
 4   unstranded          121320 non-null  int64  
 5   stranded_first      121320 non-null  int64  
 6   stranded_second     121320 non-null  int64  
 7   tpm_unstranded      121320 non-null  float64
 8   fpkm_unstranded     121320 non-null  float64
 9   fpkm_uq_unstranded  121320 non-null  float64
 10  class               121320 non-null  int64  
dtypes: float64(3), int64(5), object(3)
memory usage: 10.2+ MB


In [243]:
# Extract relevant columns
dfe=df.loc[:,['gene_name','tpm_unstranded','class']]

In [244]:
dfe.head()

,gene_name,tpm_unstranded,class
0,TSPAN6,58.4397,0
1,TNMD,2.9499,0
2,DPM1,102.1078,0
3,SCYL3,16.5362,0
4,C1orf112,4.0842,0


In [245]:
# Step 2: Preprocess the data
# Get the total number of unique genes in the dataset
total_unique_genes = len(dfe['gene_name'].unique())
print("Total number of unique genes:", total_unique_genes)

Total number of unique genes: 59427


In [246]:
# Step 3: Filter genes with TPM values present in both classes
genes_both_classes = dfe.groupby('gene_name').filter(lambda x: x['class'].nunique() == 2)

In [247]:
# Step 4: Normalize the 'tpm_unstranded' values
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
genes_both_classes['tpm_unstranded'] = scaler.fit_transform(genes_both_classes[['tpm_unstranded']])

In [248]:
# Step 1.3: Noise Reduction - Remove outliers using Z-score method
from scipy import stats
z_scores = np.abs(stats.zscore(genes_both_classes['tpm_unstranded']))
threshold = 3
genes_both_classes = genes_both_classes[(z_scores < threshold)]

In [249]:
dfe['gene_name'].value_counts()

gene_name
Y_RNA          1512
Metazoa_SRP     340
U3              100
U6               66
SNORA70          54
               ... 
CYS1              2
PLPP6             2
KLRC2             2
KLRC3             2
AP006621.6        2
Name: count, Length: 59427, dtype: int64

In [250]:
dfe[dfe['gene_name']=='PIK3CA']  # PIK3CA Gene

,gene_name,tpm_unstranded,class
5204,PIK3CA,10.6744,0
65864,PIK3CA,6.7967,1


In [251]:
dfe[dfe['gene_name']=='TP53']

,gene_name,tpm_unstranded,class
8124,TP53,55.8894,0
68784,TP53,42.4685,1


In [252]:
dfe[dfe['gene_name']=='BRCA1']

,gene_name,tpm_unstranded,class
310,BRCA1,4.8549,0
60970,BRCA1,3.0269,1


In [253]:
dfe[dfe['gene_name']=='NF1']

,gene_name,tpm_unstranded,class
17196,NF1,19.8343,0
77856,NF1,7.0352,1


In [254]:
#Label Encoding
from sklearn.preprocessing import LabelEncoder
encoder=LabelEncoder()

In [255]:
# Encode labels in column 'species'
genes_both_classes['gene_name'] = encoder.fit_transform(genes_both_classes['gene_name'])

In [256]:
from sklearn.model_selection import train_test_split

In [257]:
y = genes_both_classes['class']
X = genes_both_classes.drop(['class'], axis=1)

In [258]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)

In [259]:
# Step 6: Machine Learning Algorithm (Decision Tree Classifier)
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report

In [260]:
dtree=DecisionTreeClassifier()

In [261]:
dtree.fit(X_train,y_train)

DecisionTreeClassifier()

In [262]:
y_predictions=dtree.predict(X_test)

In [263]:
# Step 7: Evaluate accuracy
accuracy = accuracy_score(y_test, y_predictions)
print("Accuracy Score:", accuracy)

Accuracy Score: 0.3530220913917852


In [264]:
print(classification_report(y_test,y_predictions))

              precision    recall  f1-score   support

           0       0.36      0.39      0.38     18153
           1       0.34      0.32      0.33     18196

    accuracy                           0.35     36349
   macro avg       0.35      0.35      0.35     36349
weighted avg       0.35      0.35      0.35     36349



In [265]:
# Step 7: Define fitness function
from sklearn.metrics import accuracy_score

def fitness_function(selected_genes):
    X_selected = genes_both_classes[genes_both_classes['gene_name'].isin(selected_genes)]
    y = X_selected['class']
    X = X_selected.drop(['class'], axis=1)
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=101)
    dtree = DecisionTreeClassifier()
    dtree.fit(X_train, y_train)
    y_predictions = dtree.predict(X_test)
    accuracy = accuracy_score(y_test, y_predictions)
    if accuracy >= 0.85:
        return accuracy  # Achieved the goal
    else:
        return accuracy / 0.85  # Penalize if below the goal

In [266]:
# Step 8: Genetic Algorithm
def genetic_algorithm(population_size, gene_pool, fitness_func, generations):
    population = [np.random.choice(gene_pool, 30, replace=False) for _ in range(population_size)]
    for _ in range(generations):
        scores = [(genes, fitness_func(genes)) for genes in population]
        scores.sort(key=lambda x: x[1], reverse=True)
        population = [scores[i][0] for i in range(population_size)]
    return population[0]

In [267]:
# Step 9: Define gene pool
gene_pool = genes_both_classes['gene_name'].unique()

In [268]:
# Step 10: Run genetic algorithm
selected_genes = genetic_algorithm(population_size=100, gene_pool=gene_pool, fitness_func=fitness_function, generations=100)

In [269]:
# Step 11: Print selected genes
print(selected_genes)

[28191 21889 19140  7548 33482 16001 12164  6924 25603 52419 36036 28891
 34532 21609 49846 56443 26658 12643 46694 37095 59014 28855 39015 54918
   660 21099 34390 59235  1626 50661]


In [270]:
# Calculate and print new accuracy score
X_selected = genes_both_classes[genes_both_classes['gene_name'].isin(selected_genes)]
y_selected = X_selected['class']
X_selected = X_selected.drop(['class'], axis=1)
X_train, X_test, y_train, y_test = train_test_split(X_selected, y_selected, test_size=0.3, random_state=101)
dtree = DecisionTreeClassifier()
dtree.fit(X_train, y_train)
y_predictions = dtree.predict(X_test)
new_accuracy = accuracy_score(y_test, y_predictions)
print("New Accuracy Score:", new_accuracy)

New Accuracy Score: 0.5555555555555556


In [271]:
print(classification_report(y_test,y_predictions))

              precision    recall  f1-score   support

           0       0.58      0.70      0.64        10
           1       0.50      0.38      0.43         8

    accuracy                           0.56        18
   macro avg       0.54      0.54      0.53        18
weighted avg       0.55      0.56      0.54        18

